# Data preparation

## Imports

In [2]:
1 + 1

2

In [3]:
import os
import sys

sys.path.append("..")

In [4]:
import numpy as np
import pandas as pd
import dill

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [5]:
# Comment next two lines to run in collab
%load_ext autoreload
%autoreload 2

from rl_trading.features.data_processing import (
    create_reverse_fx_tickers,
)

---
## Load raw historical data

In [6]:
# comment for collab
data_folder = "C:\\Users\\Ivan\\rl_trading\\data\\"
# data_folder = ""

In [7]:
historical_data = pd.read_parquet(f"{data_folder}FX_data.parquet.gzip")

In [8]:
for col in historical_data.columns:
    if col in ["ccy", "timestamp"]:
        continue
    historical_data[col] = historical_data[col].astype(float)

In [9]:
historical_data["ccy"] = historical_data["ccy"].str.upper()

In [10]:
# historical_data = historical_data.loc[historical_data["ccy"].isin(["EURUSD"]), :]

In [11]:
historical_data["ccy"].unique()

array(['EURJPY', 'EURUSD', 'SGDJPY', 'USDJPY', 'USDSGD'], dtype=object)

---
## Add returns

In [12]:
historical_data["date"] = historical_data["timestamp"].dt.date

for lag in [1, 2, 10, 20, 30, 60, 120]:
    historical_data[f"ret_{lag}"] = historical_data.groupby(["ccy", "date"])["close"].pct_change(1).fillna(0)
    historical_data[f"ret_{lag}_sq"] = historical_data[f"ret_{lag}"] ** 2

historical_data = historical_data.drop(columns=["date"])

---
## Split into 3 parts

Leave first 3 years to train scaler

In [13]:
historical_data_scaler_train = historical_data.loc[
    historical_data["timestamp"] <= "2023-01-01 00:00:00", :
]

Take next 2 years for training

In [14]:
historical_data_train = historical_data.loc[
    (historical_data["timestamp"] >= "2023-01-02 00:00:00")
    & (historical_data["timestamp"] < "2023-12-01 00:00:00"),
    :,
]

Leave last month as validation set

In [15]:
historical_data_validate = historical_data.loc[
    (historical_data["timestamp"] >= "2023-12-01 00:00:00"), :
]

---
## Set close prices aside for state

In [16]:
def extract_close_prices(historical_data: pd.DataFrame) -> dict:
    """
    Extract close prices and transform to dict
    """
    historical_data = (
        pd.pivot_table(
            data=historical_data, index="timestamp", columns="ccy", values="close"
        )
        .ffill()
        .dropna()
    )

    historical_data = create_reverse_fx_tickers(historical_data)
    historical_data = historical_data.to_dict(orient="index")

    historical_data = {str(k): v for k, v in historical_data.items()}
    return historical_data

In [17]:
historical_prices_train = extract_close_prices(historical_data_train)
historical_prices_validate = extract_close_prices(historical_data_validate)

---
## Train and apply StandardScaler

In [18]:
normal_scaler = StandardScaler().fit(
    historical_data_scaler_train.drop(columns=["timestamp", "ccy"])
)

In [19]:
def apply_scaler(normal_scaler: StandardScaler, hist_data: pd.DataFrame):
    features = normal_scaler.transform(hist_data.set_index(["timestamp", "ccy"]))
    multi_index = pd.MultiIndex.from_frame(hist_data[["timestamp", "ccy"]])
    features = pd.DataFrame(features, index=multi_index)
    return features.unstack("ccy").ffill().copy()

In [20]:
historical_data_scaler_train_scaled = apply_scaler(
    normal_scaler, historical_data_scaler_train
)
historical_data_train_scaled = apply_scaler(normal_scaler, historical_data_train)
historical_data_validate_scaled = apply_scaler(normal_scaler, historical_data_validate)

---
## Train and apply PCA

In [21]:
pca_decomposition = PCA(n_components=26).fit(
    historical_data_scaler_train_scaled.dropna()
)

Leaving those that explain more than 0.5% of variance

Alternative approach is to leave the ones with Eigen value(`explained_variance_`) higher than 1)

In [25]:
pca_decomposition.explained_variance_

array([22.98324755, 20.55346062, 16.72710792, 10.60781705,  8.58705469,
        7.53021187,  6.09549847,  5.9055975 ,  5.01307065,  3.70834661,
        3.27609297,  3.14730401,  2.58769262,  2.32448223,  2.16184977,
        2.14454462,  1.9331662 ,  1.81570088,  1.33614645,  1.28538396,
        1.17564563,  1.04870728,  0.93518218,  0.80873567,  0.68416791,
        0.66515199])

In [26]:
pca_decomposition.explained_variance_ratio_

array([0.16071031, 0.14372003, 0.11696427, 0.07417514, 0.06004496,
       0.05265499, 0.04262276, 0.04129488, 0.03505389, 0.02593061,
       0.02290807, 0.02200752, 0.01809444, 0.01625394, 0.01511673,
       0.01499572, 0.01351766, 0.01269628, 0.009343  , 0.00898804,
       0.0082207 , 0.00733308, 0.00653926, 0.00565508, 0.00478404,
       0.00465107])

In [32]:
sum(pca_decomposition.explained_variance_ratio_)

np.float64(0.9442764937640867)

In [27]:
def apply_pca(pca_decomposition: PCA, hist_data: pd.DataFrame):
    features = pca_decomposition.transform(hist_data)
    final_dict = {}
    for i, date in enumerate(hist_data.index.to_list()):
        final_dict[str(date)] = features[i, :]
    return final_dict

In [28]:
historical_data_train_final = apply_pca(pca_decomposition, historical_data_train_scaled)
historical_data_validate_final = apply_pca(
    pca_decomposition, historical_data_validate_scaled
)

---
## Save results

In [29]:
final_data = {
    "train_prices": historical_prices_train,
    "train_data": historical_data_train_final,
    "validate_prices": historical_prices_validate,
    "validate_data": historical_data_validate_final,
}

In [30]:
data_folder = "C:\\Users\\Ivan\\rl_trading\\data\\"

In [31]:
%%time
with open(f"{data_folder}preprocessed_data.pkl", "wb") as f:
    dill.dump(final_data, f, dill.HIGHEST_PROTOCOL)

CPU times: total: 22.5 s
Wall time: 22.6 s
